# C10: City Swap for selected challenger winners

Self-contained city-swap evaluation for the selected best-per-family challenger models.

Models:
- label_smoothing_eps01_2ep
- rdrop_alpha10_2ep
- class_balanced_cbce_beta099_2ep
- bert_gdro_eta005_2ep
- focal_gamma1_2ep


In [1]:
import gc
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CWD = Path.cwd().resolve()
NOTEBOOKS_DIR = CWD.parent if CWD.name == 'challengers' else CWD
EVAL_REPO = NOTEBOOKS_DIR.parent
WORKSPACE_ROOT = EVAL_REPO.parent

def first_existing(paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)

PROJECT_ROOT_CANDIDATES = [
    WORKSPACE_ROOT / 'my-repository',
    WORKSPACE_ROOT / 'resume-screening',
    WORKSPACE_ROOT / 'resume-screening' / 'resume-screening',
]
PROJECT_ROOT = first_existing(PROJECT_ROOT_CANDIDATES)
if PROJECT_ROOT is None:
    PROJECT_ROOT = PROJECT_ROOT_CANDIDATES[0]

TEST_CSV_CANDIDATES = [
    WORKSPACE_ROOT / 'my-repository' / 'data' / 'processed' / 'test.csv',
    WORKSPACE_ROOT / 'resume-screening' / 'data' / 'processed' / 'test.csv',
    WORKSPACE_ROOT / 'resume-screening' / 'resume-screening' / 'data' / 'processed' / 'test.csv',
]
LABEL_MAP_CANDIDATES = [
    WORKSPACE_ROOT / 'my-repository' / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
    WORKSPACE_ROOT / 'resume-screening' / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
    WORKSPACE_ROOT / 'resume-screening' / 'resume-screening' / 'data' / 'processed' / 'label_to_supercategory_v1.csv',
]
MODEL_ROOT_CANDIDATES = PROJECT_ROOT_CANDIDATES
TEST_CSV = first_existing(TEST_CSV_CANDIDATES)
LABEL_MAP_CSV = first_existing(LABEL_MAP_CANDIDATES)
MODEL_ROOT = first_existing(MODEL_ROOT_CANDIDATES)
if MODEL_ROOT is None:
    MODEL_ROOT = MODEL_ROOT_CANDIDATES[0]

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'challengers_city_swap' / 'c10_selected_family_city_swap'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('MODEL_ROOT:', MODEL_ROOT)
print('TEST_CSV:', TEST_CSV)
print('LABEL_MAP_CSV:', LABEL_MAP_CSV)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('device:', device)

if TEST_CSV is None:
    raise FileNotFoundError(f'test.csv not found. Checked: {TEST_CSV_CANDIDATES}')
if LABEL_MAP_CSV is None:
    raise FileNotFoundError(f'label_to_supercategory_v1.csv not found. Checked: {LABEL_MAP_CANDIDATES}')


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
MODEL_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
TEST_CSV: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/data/processed/test.csv
LABEL_MAP_CSV: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/data/processed/label_to_supercategory_v1.csv
OUTPUT_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/results/challengers_city_swap/c10_selected_family_city_swap
device: cpu


In [2]:
SELECTED_MODELS = [
    {
        'family': 'label_smoothing',
        'model_name': 'label_smoothing_eps01_2ep',
        'path_candidates': [
            MODEL_ROOT / 'notebooks' / 'models' / 'challengers' / 'label_smoothing_eps01_2ep',
            MODEL_ROOT / 'models' / 'challengers' / 'label_smoothing_eps01_2ep',
        ],
    },
    {
        'family': 'rdrop',
        'model_name': 'rdrop_alpha10_2ep',
        'path_candidates': [
            MODEL_ROOT / 'notebooks' / 'models' / 'challengers' / 'rdrop_alpha10_2ep',
            MODEL_ROOT / 'models' / 'challengers' / 'rdrop_alpha10_2ep',
        ],
    },
    {
        'family': 'class_balanced',
        'model_name': 'class_balanced_cbce_beta099_2ep',
        'path_candidates': [
            MODEL_ROOT / 'notebooks' / 'models' / 'challengers' / 'class_balanced_cbce_beta099_2ep',
            MODEL_ROOT / 'models' / 'challengers' / 'class_balanced_cbce_beta099_2ep',
        ],
    },
    {
        'family': 'groupdro',
        'model_name': 'bert_gdro_eta005_2ep',
        'path_candidates': [
            MODEL_ROOT / 'models' / 'challengers' / 'bert_gdro_eta005_2ep',
            MODEL_ROOT / 'notebooks' / 'models' / 'challengers' / 'bert_gdro_eta005_2ep',
        ],
    },
    {
        'family': 'focal',
        'model_name': 'focal_gamma1_2ep',
        'path_candidates': [
            MODEL_ROOT / 'notebooks' / 'models' / 'challengers' / 'focal_gamma1_2ep',
            MODEL_ROOT / 'models' / 'challengers' / 'focal_gamma1_2ep',
        ],
    },
]

SWAP_CITIES = ['Москва', 'Екатеринбург', 'Новосибирск', 'Краснодар', 'Воронеж']
BATCH_SIZE = 8
MAX_LENGTH = 128

def resolve_model_path(candidates):
    return next((Path(p) for p in candidates if Path(p).exists()), None)

resolved_rows = []
for item in SELECTED_MODELS:
    resolved_path = resolve_model_path(item['path_candidates'])
    resolved_rows.append({
        'family': item['family'],
        'model_name': item['model_name'],
        'resolved_model_path': str(resolved_path) if resolved_path else '',
        'exists': bool(resolved_path),
    })

resolved_df = pd.DataFrame(resolved_rows)
resolved_df


,family,model_name,resolved_model_path,exists
0,label_smoothing,label_smoothing_eps01_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
1,rdrop,rdrop_alpha10_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
2,class_balanced,class_balanced_cbce_beta099_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
3,groupdro,bert_gdro_eta005_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
4,focal,focal_gamma1_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True


In [3]:
CITY_PATTERNS = [
    'санкт-петербург', 'нижний новгород', 'ростов-на-дону',
    'набережные челны', 'магнитогорск', 'новосибирск',
    'екатеринбург', 'красноярск', 'волгоград', 'калининград',
    'владивосток', 'хабаровск', 'ставрополь', 'саратов',
    'челябинск', 'самара', 'казань', 'москва', 'омск',
    'воронеж', 'пермь', 'тюмень', 'томск', 'уфа',
    'тольятти', 'барнаул', 'иркутск', 'пенза', 'липецк',
    'кемерово', 'сочи', 'тверь', 'минск', 'алматы',
    'симферополь', 'ярославль', 'ульяновск', 'ижевск',
    'оренбург', 'мск', 'спб', 'питер',
]
escaped = [re.escape(c) for c in CITY_PATTERNS]
CITY_RE = re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE)

def swap_cities_in_text(text, target_city):
    if pd.isna(text):
        return ''
    def replacer(match):
        orig = match.group(0)
        return target_city.capitalize() if orig and orig[0].isupper() else target_city.lower()
    return CITY_RE.sub(replacer, str(text))

def predict_batch(texts, model, tokenizer, batch_size=BATCH_SIZE):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
            preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        del enc, outputs
    return np.array(all_preds)


In [4]:
def find_eval_model_files(model_dir):
    model_dir = Path(model_dir)
    if (model_dir / 'config.json').exists():
        return 'hf', model_dir
    if (model_dir / 'bert' / 'config.json').exists() and any((model_dir / name).exists() for name in ['classifier.pt', 'model.pt']):
        return 'split_classifier', model_dir
    candidates = [sub for sub in model_dir.iterdir() if sub.is_dir() and (sub / 'config.json').exists()]
    if candidates:
        preferred = [sub for sub in candidates if sub.name == 'final']
        return 'hf', preferred[0] if preferred else sorted(candidates)[-1]
    return None, None

def _extract_state_dict(obj):
    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'classifier_state_dict']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return None

def _load_split_classifier_weights(model, model_dir):
    model_dir = Path(model_dir)
    target_state = model.classifier.state_dict()
    for candidate in [model_dir / 'classifier.pt', model_dir / 'model.pt']:
        if not candidate.exists():
            continue
        obj = torch.load(candidate, map_location='cpu')
        state = _extract_state_dict(obj)
        if not isinstance(state, dict):
            continue
        remapped = {}
        for key, value in state.items():
            norm_key = key.replace('module.', '')
            if norm_key.startswith('classifier.'):
                norm_key = norm_key[len('classifier.'):]
            if norm_key in target_state and getattr(value, 'shape', None) == target_state[norm_key].shape:
                remapped[norm_key] = value
        if set(remapped.keys()) == set(target_state.keys()):
            model.classifier.load_state_dict(remapped, strict=True)
            return True
    return False

def load_model_for_eval(model_name, model_dir):
    fmt, path = find_eval_model_files(model_dir)
    if fmt is None:
        print(f'  No model files found in {model_dir}')
        return None
    try:
        if fmt == 'split_classifier':
            bert_path = path / 'bert'
            tokenizer_path = path if (path / 'tokenizer.json').exists() else bert_path
            tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(bert_path), output_loading_info=True)
            missing_keys = set(loading_info.get('missing_keys', []))
            if any(k.startswith('classifier.') for k in missing_keys) and not _load_split_classifier_weights(model, path):
                print(f'  Could not restore classifier head for {model_name}')
                return None
        else:
            tokenizer = AutoTokenizer.from_pretrained(str(path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(path), output_loading_info=True)
            missing_keys = set(loading_info.get('missing_keys', []))
            if any(k.startswith('classifier.') for k in missing_keys):
                print(f'  Missing classifier head for {model_name}: {sorted(missing_keys)}')
                return None

        le = None
        for le_path in [path / 'label_encoder.joblib', path.parent / 'label_encoder.joblib', model_dir / 'label_encoder.joblib']:
            if le_path.exists():
                le = joblib.load(le_path)
                break
        if le is None:
            print(f'  No label encoder for {model_name}')
            return None

        model = model.to(device).eval()
        return {'model': model, 'tokenizer': tokenizer, 'label_encoder': le}
    except Exception as exc:
        print(f'  Failed to load {model_name}: {type(exc).__name__}: {exc}')
        return None


In [5]:
df_eval = pd.read_csv(TEST_CSV)
mapping = pd.read_csv(LABEL_MAP_CSV)
label_to_super = dict(zip(mapping['label'], mapping['supercategory']))

summary_rows = []

for item in SELECTED_MODELS:
    print('\n' + '=' * 80)
    print(f"Evaluating: {item['model_name']} ({item['family']})")
    model_dir = resolve_model_path(item['path_candidates'])
    row = {
        'family': item['family'],
        'model_name': item['model_name'],
        'model_path': str(model_dir) if model_dir else '',
        'accuracy': None,
        'macro_f1': None,
        'overall_flip_rate': None,
        'status': '',
        'error': '',
    }

    if model_dir is None:
        row['status'] = 'missing_model_dir'
        row['error'] = 'No matching model directory found in current workspace'
        summary_rows.append(row)
        print('  Missing model directory')
        continue

    loaded = load_model_for_eval(item['model_name'], model_dir)
    if loaded is None:
        row['status'] = 'load_failed'
        row['error'] = 'Model could not be loaded with available files'
        summary_rows.append(row)
        continue

    model = loaded['model']
    tokenizer = loaded['tokenizer']
    le = loaded['label_encoder']

    base_texts = df_eval['resume_text'].fillna('').astype(str).tolist()
    orig_preds = predict_batch(base_texts, model, tokenizer)

    any_flip = np.zeros(len(df_eval), dtype=bool)
    per_city = {}

    for swap_city in SWAP_CITIES:
        swapped = df_eval['resume_text'].apply(lambda x: swap_cities_in_text(x, swap_city)).fillna('').astype(str).tolist()
        swap_preds = predict_batch(swapped, model, tokenizer)
        flipped = swap_preds != orig_preds
        flip_rate = float(flipped.mean())
        any_flip |= flipped
        per_city[swap_city] = flip_rate
        row[f'flip_{swap_city}'] = flip_rate
        print(f"  {swap_city}: {flip_rate:.3f} ({int(flipped.sum())}/{len(df_eval)})")
        del swapped, swap_preds, flipped
        gc.collect()

    y_true = le.transform(df_eval['label'].map(label_to_super).fillna('generic_it_ops'))
    acc = float(accuracy_score(y_true, orig_preds))
    f1 = float(f1_score(y_true, orig_preds, average='macro'))
    overall_flip = float(any_flip.mean())

    row['accuracy'] = acc
    row['macro_f1'] = f1
    row['overall_flip_rate'] = overall_flip
    row['status'] = 'ok'
    print(f"  Acc={acc:.3f}  F1={f1:.3f}  Flip={overall_flip:.3f}")

    model_output_dir = OUTPUT_DIR / item['model_name']
    model_output_dir.mkdir(parents=True, exist_ok=True)
    (model_output_dir / 'city_swap_summary.json').write_text(json.dumps({
        'family': item['family'],
        'model_name': item['model_name'],
        'model_path': str(model_dir),
        'accuracy': acc,
        'macro_f1': f1,
        'overall_flip_rate': overall_flip,
        'per_swap_city': {city: {'flip_rate': rate} for city, rate in per_city.items()},
    }, indent=2, ensure_ascii=False))

    summary_rows.append(row)

    del model, tokenizer, le, orig_preds, any_flip
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'c10_selected_family_city_swap_summary.csv', index=False)
summary_df



Evaluating: label_smoothing_eps01_2ep (label_smoothing)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 446.24it/s, Materializing param=classifier.weight]                                      


  Москва: 0.021 (118/5510)
  Екатеринбург: 0.045 (248/5510)
  Новосибирск: 0.039 (215/5510)
  Краснодар: 0.051 (280/5510)
  Воронеж: 0.041 (226/5510)
  Acc=0.608  F1=0.623  Flip=0.089

Evaluating: rdrop_alpha10_2ep (rdrop)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1449.38it/s, Materializing param=classifier.weight]                                     


  Москва: 0.014 (76/5510)
  Екатеринбург: 0.039 (217/5510)
  Новосибирск: 0.027 (151/5510)
  Краснодар: 0.029 (159/5510)
  Воронеж: 0.023 (129/5510)
  Acc=0.607  F1=0.620  Flip=0.062

Evaluating: class_balanced_cbce_beta099_2ep (class_balanced)


...

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1766.83it/s, Materializing param=classifier.weight]                                      


  Москва: 0.016 (88/5510)
  Екатеринбург: 0.044 (241/5510)
  Новосибирск: 0.036 (196/5510)
  Краснодар: 0.033 (182/5510)
  Воронеж: 0.048 (266/5510)
  Acc=0.575  F1=0.590  Flip=0.087

Evaluating: focal_gamma1_2ep (focal)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1926.95it/s, Materializing param=classifier.weight]                                      


  Москва: 0.020 (112/5510)
  Екатеринбург: 0.047 (261/5510)
  Новосибирск: 0.049 (271/5510)
  Краснодар: 0.041 (224/5510)
  Воронеж: 0.031 (171/5510)
  Acc=0.604  F1=0.619  Flip=0.087


,family,model_name,model_path,accuracy,macro_f1,overall_flip_rate,status,error,flip_Москва,flip_Екатеринбург,flip_Новосибирск,flip_Краснодар,flip_Воронеж
0,label_smoothing,label_smoothing_eps01_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,0.608348,0.622958,0.089111,ok,,0.021416,0.045009,0.039020,0.050817,0.041016
1,rdrop,rdrop_alpha10_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,0.606715,0.620074,0.062069,ok,,0.013793,0.039383,0.027405,0.028857,0.023412
2,class_balanced,class_balanced_cbce_beta099_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,0.606897,0.620787,0.057713,ok,,0.013793,0.032849,0.028857,0.028857,0.026134
3,groupdro,bert_gdro_eta005_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,0.574773,0.590127,0.087477,ok,,0.015971,0.043739,0.035572,0.033031,0.048276
4,focal,focal_gamma1_2ep,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,0.604174,0.619110,0.086751,ok,,0.020327,0.047368,0.049183,0.040653,0.031034


In [6]:
lines = ['=== C10 selected family city-swap summary ===']
for _, row in summary_df.iterrows():
    lines.extend([
        f"family: {row['family']}",
        f"model_name: {row['model_name']}",
        f"status: {row['status']}",
        f"accuracy: {row['accuracy']}",
        f"macro_f1: {row['macro_f1']}",
        f"overall_flip_rate: {row['overall_flip_rate']}",
        f"model_path: {row['model_path']}",
        f"error: {row['error'] if row['error'] else '-'}",
        '-' * 60,
    ])

report_text = '\n'.join(lines)
(OUTPUT_DIR / 'c10_selected_family_city_swap_report.txt').write_text(report_text)
print(report_text)


=== C10 selected family city-swap summary ===
family: label_smoothing
model_name: label_smoothing_eps01_2ep
status: ok
accuracy: 0.6083484573502722
macro_f1: 0.6229579754516105
overall_flip_rate: 0.08911070780399274
model_path: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers/label_smoothing_eps01_2ep
error: -
------------------------------------------------------------
family: rdrop
model_name: rdrop_alpha10_2ep
status: ok
accuracy: 0.6067150635208711
macro_f1: 0.6200736985847453
overall_flip_rate: 0.06206896551724138
model_path: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers/rdrop_alpha10_2ep
error: -
------------------------------------------------------------
family: class_balanced
model_name: class_balanced_cbce_beta099_2ep
status: ok
accuracy: 0.6068965517241379
macro_f1: 0.6207870115762675
overall_flip_rate: 0.05771324863883848
model_path: /Users/natashaagapova/Documents/A-INNOPOLIS/A